In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
train_path = 'Training/'
test_path = 'Testing/'
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']
img_size = (64, 64)   # keep modest for baseline

# Image loading function
def load_images_from_folder(base_path, classes, img_size=(64, 64)):
    X = []
    y = []

    for cls in classes:
        folder = os.path.join(base_path, cls)
        if not os.path.exists(folder):
            print(f"Warning: folder not found -> {folder}")
            continue

        files = [f for f in os.listdir(folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for fname in files:
            img_path = os.path.join(folder, fname)
            try:
                img = Image.open(img_path).convert('L')   # force grayscale
                img = img.resize(img_size)
                img_array = np.asarray(img, dtype=np.float32) / 255.0
                img_array = img_array.reshape(-1)         # flatten to 1D vector

                # sanity check
                if img_array.shape[0] != img_size[0] * img_size[1]:
                    print(f"Skipping malformed image: {img_path}")
                    continue

                X.append(img_array)
                y.append(cls)

            except Exception as e:
                print(f"Skipping {img_path}: {e}")

    # force a true numeric 2D matrix
    X = np.stack(X).astype(np.float32)
    y = np.array(y, dtype=str)

    return X, y

# Load training and testing data
X_train, y_train = load_images_from_folder(train_path, classes, img_size=img_size)
X_test, y_test = load_images_from_folder(test_path, classes, img_size=img_size)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Training labels distribution:")
print(pd.Series(y_train).value_counts())
print("\nTesting labels distribution:")
print(pd.Series(y_test).value_counts())

X_train shape: (5600, 4096)
X_test shape: (1600, 4096)
Training labels distribution:
glioma        1400
meningioma    1400
notumor       1400
pituitary     1400
Name: count, dtype: int64

Testing labels distribution:
glioma        400
meningioma    400
notumor       400
pituitary     400
Name: count, dtype: int64


In [3]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='macro')

print("Random Forest Results")
print("Accuracy:", rf_acc)
print("Macro F1:", rf_f1)
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Results
Accuracy: 0.87
Macro F1: 0.8657456837414703

Classification Report:
              precision    recall  f1-score   support

      glioma       0.94      0.66      0.78       400
  meningioma       0.83      0.88      0.85       400
     notumor       0.82      1.00      0.90       400
   pituitary       0.93      0.94      0.94       400

    accuracy                           0.87      1600
   macro avg       0.88      0.87      0.87      1600
weighted avg       0.88      0.87      0.87      1600



In [4]:
print("X_train:", X_train.dtype, X_train.shape, np.isnan(X_train).any(), np.isinf(X_train).any())
print("X_test:", X_test.dtype, X_test.shape, np.isnan(X_test).any(), np.isinf(X_test).any())

print("y_train type:", type(y_train), "dtype:", y_train.dtype, "shape:", y_train.shape)
print("y_test type:", type(y_test), "dtype:", y_test.dtype, "shape:", y_test.shape)

print("Unique y_train labels:", np.unique(y_train))
print("Unique y_test labels:", np.unique(y_test))

y_train = np.asarray(y_train).astype(str)
y_test = np.asarray(y_test).astype(str)

X_train: float32 (5600, 4096) False False
X_test: float32 (1600, 4096) False False
y_train type: <class 'numpy.ndarray'> dtype: <U10 shape: (5600,)
y_test type: <class 'numpy.ndarray'> dtype: <U10 shape: (1600,)
Unique y_train labels: ['glioma' 'meningioma' 'notumor' 'pituitary']
Unique y_test labels: ['glioma' 'meningioma' 'notumor' 'pituitary']


In [5]:
# encode labels
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

# scale inputs for MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

mlp_model.fit(X_train_scaled, y_train_enc)
mlp_pred_enc = mlp_model.predict(X_test_scaled)

# convert predictions back to class names if needed
mlp_pred = label_encoder.inverse_transform(mlp_pred_enc)

print("Accuracy:", accuracy_score(y_test, mlp_pred))
print("Macro F1:", f1_score(y_test, mlp_pred, average='macro'))
print(classification_report(y_test, mlp_pred))

Accuracy: 0.8775
Macro F1: 0.8751208684485099
              precision    recall  f1-score   support

      glioma       0.88      0.74      0.80       400
  meningioma       0.84      0.85      0.84       400
     notumor       0.87      0.99      0.93       400
   pituitary       0.92      0.94      0.93       400

    accuracy                           0.88      1600
   macro avg       0.88      0.88      0.88      1600
weighted avg       0.88      0.88      0.88      1600



In [6]:
print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("RF Macro F1:", f1_score(y_test, rf_pred, average='macro'))

print("MLP Accuracy:", accuracy_score(y_test, mlp_pred))
print("MLP Macro F1:", f1_score(y_test, mlp_pred, average='macro'))

RF Accuracy: 0.87
RF Macro F1: 0.8657456837414703
MLP Accuracy: 0.8775
MLP Macro F1: 0.8751208684485099
